In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
spark = SparkSession.builder \
    .appName("Trade Analytics") \
    .getOrCreate()

In [ ]:
df = spark.read.csv(
    "/content/hormuz_trade_tier_continental_2026.csv",
    header=True,
    inferSchema=True
)

In [ ]:
df.show(5)

+----------+-------------+---------+------+----------+--------------+--------+--------+---------+-----------+--------------------+-------------------+---------------------------+------------------+------------+-----------------+-------------------+----------------------+--------------------+-------------------------+-----------------------------+---------+--------------------------+
|      date|  vessel_name|     mmsi|  flag|trade_tier|transit_status|toll_usd|rerouted|commodity|destination|        payment_rail|ship_hull_value_usd|insurance_premium_delta_pct|insurance_cost_usd|days_delayed|extra_fuel_tonnes|reroute_penalty_usd|total_transit_cost_usd| naval_escort_status|estimated_cargo_value_usd|total_asset_value_at_risk_usd|continent|inflation_premium_per_unit|
+----------+-------------+---------+------+----------+--------------+--------+--------+---------+-----------+--------------------+-------------------+---------------------------+------------------+------------+-----------------+

In [ ]:
df.columns

['date',
 'vessel_name',
 'mmsi',
 'flag',
 'trade_tier',
 'transit_status',
 'toll_usd',
 'rerouted',
 'commodity',
 'destination',
 'payment_rail',
 'ship_hull_value_usd',
 'insurance_premium_delta_pct',
 'insurance_cost_usd',
 'days_delayed',
 'extra_fuel_tonnes',
 'reroute_penalty_usd',
 'total_transit_cost_usd',
 'naval_escort_status',
 'estimated_cargo_value_usd',
 'total_asset_value_at_risk_usd',
 'continent',
 'inflation_premium_per_unit']

In [ ]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- mmsi: integer (nullable = true)
 |-- flag: string (nullable = true)
 |-- trade_tier: string (nullable = true)
 |-- transit_status: string (nullable = true)
 |-- toll_usd: integer (nullable = true)
 |-- rerouted: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- payment_rail: string (nullable = true)
 |-- ship_hull_value_usd: integer (nullable = true)
 |-- insurance_premium_delta_pct: double (nullable = true)
 |-- insurance_cost_usd: integer (nullable = true)
 |-- days_delayed: integer (nullable = true)
 |-- extra_fuel_tonnes: integer (nullable = true)
 |-- reroute_penalty_usd: integer (nullable = true)
 |-- total_transit_cost_usd: integer (nullable = true)
 |-- naval_escort_status: string (nullable = true)
 |-- estimated_cargo_value_usd: integer (nullable = true)
 |-- total_asset_value_at_risk_usd: integer (nullable = true)
 |-- continen

In [ ]:
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show()

+----+-----------+----+----+----------+--------------+--------+--------+---------+-----------+------------+-------------------+---------------------------+------------------+------------+-----------------+-------------------+----------------------+-------------------+-------------------------+-----------------------------+---------+--------------------------+
|date|vessel_name|mmsi|flag|trade_tier|transit_status|toll_usd|rerouted|commodity|destination|payment_rail|ship_hull_value_usd|insurance_premium_delta_pct|insurance_cost_usd|days_delayed|extra_fuel_tonnes|reroute_penalty_usd|total_transit_cost_usd|naval_escort_status|estimated_cargo_value_usd|total_asset_value_at_risk_usd|continent|inflation_premium_per_unit|
+----+-----------+----+----+----------+--------------+--------+--------+---------+-----------+------------+-------------------+---------------------------+------------------+------------+-----------------+-------------------+----------------------+-------------------+--------

In [ ]:
df = df.dropna()

In [ ]:
df = df.dropDuplicates()

In [ ]:
df = df.withColumn(
    "date",
    to_date(col("date"), "yyyy-MM-dd")
)

In [ ]:
df = df.withColumn(
    "year",
    year(col("date"))
)

In [ ]:
df = df.withColumn(
    "month",
    month(col("date"))
)

In [15]:
df.describe().show()

+-------+------------------+--------------------+-----+----------+--------------------+-----------------+--------------------+--------------+-----------+--------------------+-------------------+---------------------------+------------------+-----------------+-----------------+-------------------+----------------------+--------------------+-------------------------+-----------------------------+---------+--------------------------+------+------------------+
|summary|       vessel_name|                mmsi| flag|trade_tier|      transit_status|         toll_usd|            rerouted|     commodity|destination|        payment_rail|ship_hull_value_usd|insurance_premium_delta_pct|insurance_cost_usd|     days_delayed|extra_fuel_tonnes|reroute_penalty_usd|total_transit_cost_usd| naval_escort_status|estimated_cargo_value_usd|total_asset_value_at_risk_usd|continent|inflation_premium_per_unit|  year|             month|
+-------+------------------+--------------------+-----+----------+------------

In [17]:
top_flags = df.groupBy("flag") \
    .agg(
        sum("total_transit_cost_usd").alias("total_transit_cost")
    ) \
    .orderBy(
        col("total_transit_cost").desc()
    )

In [18]:
top_flags.show()

+----------------+------------------+
|            flag|total_transit_cost|
+----------------+------------------+
|          Turkey|         338675000|
|          Norway|         230378000|
|           India|         230087000|
|          Israel|         219407000|
|         Liberia|         219103000|
|          Panama|         206876000|
|              UK|         195309000|
|             UAE|         186794000|
|         Germany|         182534000|
|Marshall Islands|         146228000|
|             USA|         124618000|
|          Greece|         104826000|
|            Iran|          16310000|
|           China|           7715000|
|          Russia|           6195000|
|        Pakistan|           5840000|
|            Iraq|           4140000|
+----------------+------------------+



In [19]:
top_flags.toPandas().to_csv(
    "top_flags.csv",
    index=False
)

In [20]:
continent_summary = df.groupBy("continent") \
    .agg(
        sum("total_transit_cost_usd").alias("continent_trade")
    )

In [21]:
continent_summary.show()

+-------------+---------------+
|    continent|continent_trade|
+-------------+---------------+
|       Europe|      713047000|
|      Eurasia|        6195000|
|       Africa|      219103000|
|North America|      331494000|
|      Oceania|      146228000|
|         Asia|     1008968000|
+-------------+---------------+



In [22]:
continent_summary.toPandas().to_csv(
    "continent_summary.csv",
    index=False
)

In [23]:
trade_trend = df.groupBy("year", "month") \
    .agg(
        sum("total_transit_cost_usd").alias("monthly_transit_cost")
    ) \
    .orderBy("year", "month")

In [24]:
trade_trend.show()

+----+-----+--------------------+
|year|month|monthly_transit_cost|
+----+-----+--------------------+
|2026|    3|             6650000|
|2026|    4|          1119688000|
|2026|    5|          1298697000|
+----+-----+--------------------+



In [25]:
trade_trend.toPandas().to_csv(
    "trade_trend.csv",
    index=False
)

In [26]:
flags_pd = top_flags.toPandas()

continent_pd = continent_summary.toPandas()

trend_pd = trade_trend.toPandas()

In [27]:
fig = px.bar(
    flags_pd.head(10),
    x="flag",
    y="total_transit_cost",
    title="Top Flags by Transit Cost"
)

fig.show()

In [28]:
fig2 = px.pie(
    continent_pd,
    names="continent",
    values="continent_trade",
    title="Continent-wise Transit Cost"
)

fig2.show()

In [29]:
fig3 = px.line(
    trend_pd,
    x="month",
    y="monthly_transit_cost",
    color="year",
    title="Monthly Transit Cost Trend"
)

fig3.show()

In [30]:
ml_df = trade_trend.toPandas()

In [31]:
X = ml_df[["month"]]

y = ml_df["monthly_transit_cost"]

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [33]:
model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

In [34]:
predictions = model.predict(X_test)

In [35]:
mse = mean_squared_error(
    y_test,
    predictions
)

print("MSE:", mse)

MSE: 8.72410172841e+17


In [36]:
df.toPandas().to_csv(
    "cleaned_trade_data.csv",
    index=False
)